# 19 — Prompting for Coding Agents

    ## Scenario and success criteria

    A coding agent receives a one-file authentication fix with an exact verification command and no production authority.

    This guided lab succeeds when its assertions pass and the learner can explain why the baseline fails, what the mitigation changes, and which production controls remain outside the simulation.

    ## Learning objectives

    - Write a software-change contract.
- Enforce read, write, network, and command scope.
- Require tests and diff evidence before completion.

    **Prerequisites:** Courses 01–13 and the preceding advanced/enterprise lesson.
    **Safety boundary:** all behavior is deterministic and synthetic; there are no credentials, external calls, or side effects. Printed results are simulation evidence, not a live-model benchmark.

## Mental model and architecture

![Course 19 architecture](diagram-1.svg)

Treat the model as one uncertain component inside a deterministic control plane. Inputs, schemas, identity, authorization, metrics, release gates, and state transitions remain application responsibilities.

## Baseline and failure injection

A prompt saying 'stay in scope' is guidance, not a sandbox or policy boundary.

The next cell defines the synthetic fixture and the smallest reusable primitive needed to make that failure observable.

In [ ]:
from lab19 import ChangeContract, ProposedChange, completion_allowed, validate_plan

contract = ChangeContract(
    problem="Correct the token expiration default from zero to 3600 seconds.",
    allowed_files=("src/auth.py",),
    test_command=("pytest", "tests/test_auth.py"),
    acceptance=("auth tests pass", "only src/auth.py changes"),
)

## Experiment

Run the baseline and candidate on the same fixture so the comparison is attributable.

In [ ]:
unsafe = ProposedChange(
    files_to_read=("src/auth.py", "secrets.env"),
    files_to_write=("src/auth.py", "deploy.yml"),
    command=("curl", "production"),
)
safe = ProposedChange(
    files_to_read=("src/auth.py",),
    files_to_write=("src/auth.py",),
    command=("pytest", "tests/test_auth.py"),
)
print("unsafe violations", validate_plan(contract, unsafe))
print("safe violations", validate_plan(contract, safe))

## Evaluation

The assertions below are the executable contract. They validate both a positive path and a boundary or failure path; a printed claim alone is not proof.

In [ ]:
assert set(validate_plan(contract, unsafe)) == {"read_scope", "write_scope", "command_not_approved", "network_not_approved"}
assert validate_plan(contract, safe) == ()
assert completion_allowed(contract, tests_passed=True, diff_files=("src/auth.py",))

## Production upgrade

Run agents in isolated worktrees or sandboxes, grant least privilege, inspect untrusted issues and repository text as data, and verify the actual diff and tests. Production deployment remains a separately authorized action.

| Teaching lab | Production system |
| --- | --- |
| Synthetic fixtures | Versioned, reviewed, privacy-safe datasets |
| Deterministic simulation | Provider adapter plus optional recorded replay |
| In-process state | Durable state with tenant and retention boundaries |
| Assertions | CI gates, staged rollout, monitoring, and rollback |

## Exercises

1. Add one normal, one boundary, and one adversarial case without weakening an invariant.
2. Change one design variable and report the metric numerator, denominator, unit, and direction.
3. Write a production decision memo that identifies owner, failure policy, monitoring signal, and rollback trigger.

## Takeaway

Use probabilistic components for bounded interpretation; use trusted deterministic code for permissions, validation, metrics, and consequential state changes.